Created by Claude Opus 4.6 3/13/26 prompted and edited by Breschine Cummins.

# Interactive LDA Class Separation Explorer

**M508 — Mathematics for Machine Learning**

This notebook lets you explore how the generalized Rayleigh quotient
$$\rho(\vec{u}) = \frac{\vec{u}^T S_b\,\vec{u}}{\vec{u}^T S_w\,\vec{u}}$$
responds to changes in class geometry. Use the sliders to move class means
and adjust within-class spread, then watch the LDA discriminant direction update.

In [1]:
%matplotlib inline
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from ipywidgets import interact, FloatSlider
import warnings
warnings.filterwarnings('ignore')

In [2]:
def generate_data(mu1, mu2, mu3, sigma):
    """Generate three 2D Gaussian clusters with given means and shared spread."""
    rng = np.random.default_rng(42)
    n_per_class = 50
    cov = sigma**2 * np.eye(2)
    X1 = rng.multivariate_normal(mu1, cov, n_per_class)
    X2 = rng.multivariate_normal(mu2, cov, n_per_class)
    X3 = rng.multivariate_normal(mu3, cov, n_per_class)
    return X1, X2, X3


def compute_scatter_matrices(X1, X2, X3):
    """Compute S_w, S_b, and the overall mean."""
    classes = [X1, X2, X3]
    m = sum(len(Xi) for Xi in classes)
    n = X1.shape[1]
    x_bar = sum(Xi.sum(axis=0) for Xi in classes) / m

    S_w = np.zeros((n, n))
    S_b = np.zeros((n, n))
    for Xi in classes:
        mi = len(Xi)
        ci = Xi.mean(axis=0)
        Xi_c = Xi - ci  # center within class
        S_w += Xi_c.T @ Xi_c
        diff = (ci - x_bar).reshape(-1, 1)
        S_b += mi * (diff @ diff.T)

    return S_w, S_b, x_bar


def solve_lda(S_b, S_w, alpha=1e-4):
    """Solve the generalized eigenproblem S_b u = lambda S_w u.

    Regularize S_w slightly to ensure positive definiteness.
    Returns eigenvalues (descending) and corresponding eigenvectors.
    """
    S_w_reg = S_w + alpha * np.eye(S_w.shape[0])
    eigvals, eigvecs = eigh(S_b, S_w_reg)
    # eigh returns ascending order; flip to descending
    idx = np.argsort(eigvals)[::-1]
    return eigvals[idx], eigvecs[:, idx]


def draw_direction_arrow(ax, origin, direction, length=6.0,
                         color='black', lw=2.5, linestyle='-'):
    """Draw a long bidirectional line with arrowhead through `origin`.

    Uses FancyArrowPatch so the arrow renders reliably at any zoom.
    The arrow extends `length` data units in each direction from `origin`.
    """
    d = direction / np.linalg.norm(direction)  # unit vector
    tail = origin - length * d
    head = origin + length * d
    arrow = FancyArrowPatch(
        posA=tuple(tail), posB=tuple(head),
        arrowstyle='->', mutation_scale=18,
        color=color, linewidth=lw, linestyle=linestyle,
    )
    ax.add_patch(arrow)

## Demo 1: Moving Class Means

Class 1 is fixed at the origin. Use the sliders to move Classes 2 and 3.
Observe how both LDA discriminant directions (black and gray arrows) change to
maximize class separation relative to within-class spread.

In [3]:
def plot_lda_means(mu2_x=3.0, mu2_y=0.0, mu3_x=0.0, mu3_y=3.0, sigma=0.8):
    mu1 = np.array([0.0, 0.0])
    mu2 = np.array([mu2_x, mu2_y])
    mu3 = np.array([mu3_x, mu3_y])

    X1, X2, X3 = generate_data(mu1, mu2, mu3, sigma)
    S_w, S_b, x_bar = compute_scatter_matrices(X1, X2, X3)
    eigvals, eigvecs = solve_lda(S_b, S_w)

    u1 = eigvecs[:, 0]
    u2 = eigvecs[:, 1]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Left: scatter plot with both LDA directions
    ax = axes[0]
    ax.scatter(X1[:, 0], X1[:, 1], c='tab:blue', alpha=0.5, s=20, label='Class 1')
    ax.scatter(X2[:, 0], X2[:, 1], c='tab:orange', alpha=0.5, s=20, label='Class 2')
    ax.scatter(X3[:, 0], X3[:, 1], c='tab:green', alpha=0.5, s=20, label='Class 3')

    draw_direction_arrow(ax, x_bar, u1, length=6.0, color='black', lw=2.5)
    draw_direction_arrow(ax, x_bar, u2, length=6.0, color='gray', lw=2.0,
                         linestyle='--')
    ax.plot(*x_bar, 'kx', markersize=10, markeredgewidth=2)

    ax.set_xlim(-5, 8)
    ax.set_ylim(-5, 8)
    ax.set_aspect('equal')
    ax.legend(loc='upper left', fontsize=9)
    ax.set_title(r'Black: $\vec{q}_1$, Gray: $\vec{q}_2$' + '\n'
                 r'$\lambda_1 = $' + f'{eigvals[0]:.2f}, '
                 r'$\lambda_2 = $' + f'{eigvals[1]:.2f}',
                 fontsize=11)
    ax.set_xlabel(r'$x_1$')
    ax.set_ylabel(r'$x_2$')
    ax.grid(True, alpha=0.3)

    # Middle: projection onto q1
    ax2 = axes[1]
    for Xi, color, label in [(X1, 'tab:blue', 'Class 1'),
                              (X2, 'tab:orange', 'Class 2'),
                              (X3, 'tab:green', 'Class 3')]:
        ax2.hist(Xi @ u1, bins=15, alpha=0.5, color=color, label=label, density=True)
    ax2.set_title(r'Projection onto $\vec{q}_1$' + '\n'
                  r'$\lambda_1 = $' + f'{eigvals[0]:.2f}',
                  fontsize=11)
    ax2.set_xlabel(r'$\vec{q}_1^{\,T} \vec{x}$')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    # Right: projection onto q2
    ax3 = axes[2]
    for Xi, color, label in [(X1, 'tab:blue', 'Class 1'),
                              (X2, 'tab:orange', 'Class 2'),
                              (X3, 'tab:green', 'Class 3')]:
        ax3.hist(Xi @ u2, bins=15, alpha=0.5, color=color, label=label, density=True)
    ax3.set_title(r'Projection onto $\vec{q}_2$' + '\n'
                  r'$\lambda_2 = $' + f'{eigvals[1]:.2f}',
                  fontsize=11)
    ax3.set_xlabel(r'$\vec{q}_2^{\,T} \vec{x}$')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interact(
    plot_lda_means,
    mu2_x=FloatSlider(value=3.0, min=-3.0, max=6.0, step=0.2,
                      description='mu2 x', style={'description_width': 'initial'}),
    mu2_y=FloatSlider(value=0.0, min=-3.0, max=6.0, step=0.2,
                      description='mu2 y', style={'description_width': 'initial'}),
    mu3_x=FloatSlider(value=0.0, min=-3.0, max=6.0, step=0.2,
                      description='mu3 x', style={'description_width': 'initial'}),
    mu3_y=FloatSlider(value=3.0, min=-3.0, max=6.0, step=0.2,
                      description='mu3 y', style={'description_width': 'initial'}),
    sigma=FloatSlider(value=0.8, min=0.2, max=3.0, step=0.1,
                      description='sigma (spread)', style={'description_width': 'initial'}),
);

interactive(children=(FloatSlider(value=3.0, description='mu2 x', max=6.0, min=-3.0, step=0.2, style=SliderSty…

### Things to try

1. **Collinear means:** Set $\mu_2 = (3, 0)$ and $\mu_3 = (6, 0)$. All three means are on the $x_1$-axis. What happens to $\lambda_2$? Why?

2. **Large spread:** Crank $\sigma$ up to 2.5. What happens to $\lambda_1$? Why does the Rayleigh quotient shrink?

3. **Overlapping classes:** Move $\mu_2$ close to $(0, 0)$. The LDA direction should rotate to focus on separating Class 3 from the merged cluster.